# 流水特征工程（稳健版，无时间窗）

**目标**：从 `train_bank_statement.csv` / `testaa_bank_statement.csv` 生成
`train_statement_feature_v2.csv` / `testaa_statement_feature_v2.csv`。

**设计原则**：
1. 不做 30/60/90 天固定时间窗聚合，避免“毒特征”。
2. 仅基于每个样本自身的 `record_time` 计算相对时间差。
3. 缺失流水以 `has_statement` 指示并做稳健填充。

运行前可根据需要修改下方“路径配置”。

In [5]:
# 安装依赖（如本地环境尚未安装，可取消注释）
# !pip install pandas numpy -q

In [6]:
import os
import numpy as np
import pandas as pd
SEED = 42
np.random.seed(SEED)

In [7]:
# ===== 路径配置（请按实际环境修改） =====
TRAIN_CSV = 'train/train.csv'
TRAIN_STMT_CSV = 'train/train_bank_statement.csv'
TEST_CSV = 'testaa/testaa.csv'              # 如果没有可置为 None
TEST_STMT_CSV = 'testaa/testaa_bank_statement.csv'  # 如果没有可置为 None

OUT_TRAIN_FEAT = 'train/train_statement_feature_v2.csv'
OUT_TEST_FEAT  = 'testaa/testaa_statement_feature_v2.csv'

In [8]:
def _coerce_numeric(series):
    return pd.to_numeric(series, errors='coerce')

def _safe_div(a, b):
    b = np.where(b == 0, np.nan, b)
    return a / b

def _prep_stmt(stmt: pd.DataFrame) -> pd.DataFrame:
    df = stmt.copy()
    for col in df.columns:
        if df[col].dtype == 'O':
            df[col] = df[col].replace(['', ' ', 'nan', 'NaN', 'NULL', 'None'], np.nan)
    df['id'] = _coerce_numeric(df['id']).astype('Int64')
    df['time'] = _coerce_numeric(df['time']).astype('Int64')
    df['direction'] = _coerce_numeric(df['direction']).astype('Int8')
    df['amount'] = _coerce_numeric(df['amount']).astype(float)
    df = df.dropna(subset=['id','time','direction','amount'])
    df['abs_amount'] = df['amount'].abs()
    df['is_income'] = (df['direction']==0).astype('Int8')
    df['is_expense'] = (df['direction']==1).astype('Int8')
    df['date'] = pd.to_datetime(df['time'].astype('int64'), unit='s').dt.floor('D')
    return df

def build_statement_features(stmt: pd.DataFrame, base_df: pd.DataFrame, id_col='id', record_col='record_time') -> pd.DataFrame:
    """严格以 record_time 为快照，只用 time <= record_time 的流水来聚合，杜绝“看见未来”."""
    if stmt is None or len(stmt) == 0:
        return pd.DataFrame({id_col: base_df[id_col].unique()})

    # 预处理 + 与锚点对齐
    anchor = base_df[[id_col, record_col]].drop_duplicates()
    df = _prep_stmt(stmt).merge(anchor, on=id_col, how='inner')

    # 关键：仅保留 <= record_time 的交易，彻底避免未来信息渗入
    df = df[df['time'] <= df[record_col]]

    # === 聚合（全部基于已过滤的 df） ===
    g = df.groupby(id_col, observed=True)
    basic = g.agg(
        tx_count=('amount', 'size'),
        income_count=('is_income', 'sum'),
        expense_count=('is_expense', 'sum'),
        sum_income=('amount', lambda s: s[df.loc[s.index, 'is_income'] == 1].sum()),
        sum_expense_abs=('amount', lambda s: s[df.loc[s.index, 'is_expense'] == 1].abs().sum()),
        mean_abs_amount=('abs_amount', 'mean'),
        std_abs_amount=('abs_amount', 'std'),
        min_abs_amount=('abs_amount', 'min'),
        max_abs_amount=('abs_amount', 'max'),
        active_days=('date', 'nunique'),
        max_time=('time', 'max'),
        min_time=('time', 'min')
    ).reset_index()

    # 分位数（高效写法）
    q = df.groupby(id_col)['abs_amount'].quantile([0.5, 0.75, 0.9]).unstack()
    q.columns = ['amt_p50', 'amt_p75', 'amt_p90']
    q = q.reset_index()

    feat = pd.merge(basic, q, on=id_col, how='left')

    # 衍生比值
    feat['sum_income'] = feat['sum_income'].fillna(0.0)
    feat['sum_expense_abs'] = feat['sum_expense_abs'].fillna(0.0)
    feat['net_flow'] = feat['sum_income'] - feat['sum_expense_abs']
    feat['income_expense_ratio'] = _safe_div(feat['sum_income'], feat['sum_expense_abs'])
    feat['tx_per_active_day'] = _safe_div(feat['tx_count'], feat['active_days'].replace(0, np.nan))

    # 与锚点合并计算相对天数（仍然只基于 <= record_time 的 max/min_time）
    feat = feat.merge(anchor, on=id_col, how='right')  # 确保所有 id 都在
    delta_last = (feat[record_col].astype('float64') - feat['max_time'].astype('float64')) / 86400.0
    delta_first = (feat[record_col].astype('float64') - feat['min_time'].astype('float64')) / 86400.0
    feat['last_tx_days_before_record'] = np.where(np.isfinite(delta_last), np.maximum(delta_last, 0.0), np.nan)
    feat['first_tx_days_before_record'] = np.where(np.isfinite(delta_first), np.maximum(delta_first, 0.0), np.nan)
    feat['span_days'] = (feat['max_time'].astype('float64') - feat['min_time'].astype('float64')) / 86400.0
    feat['span_days'] = feat['span_days'].clip(lower=0.0)

    # 是否存在“快照前”的流水
    feat['has_statement'] = np.where(feat['tx_count'].fillna(0) > 0, 1, 0).astype(int)

    # 清理中间列
    feat = feat.drop(columns=['max_time', 'min_time'])

    # 数值特征缺失用中位数填充（id 不动）
    num_cols = feat.select_dtypes(include=[np.number]).columns.tolist()
    num_cols = [c for c in num_cols if c != id_col]
    med = feat[num_cols].median()
    feat[num_cols] = feat[num_cols].fillna(med)

    return feat


In [9]:
# === 运行：生成 v2 流水特征 ===
df_train = pd.read_csv(TRAIN_CSV)
feat_train = build_statement_features(pd.read_csv(TRAIN_STMT_CSV), df_train, 'id','record_time')
feat_train.to_csv(OUT_TRAIN_FEAT, index=False, encoding='utf-8')
print('train_statement_feature_v2.csv 保存完成：', OUT_TRAIN_FEAT, feat_train.shape)

if TEST_CSV is not None and os.path.exists(TEST_CSV):
    df_test = pd.read_csv(TEST_CSV)
    if TEST_STMT_CSV is not None and os.path.exists(TEST_STMT_CSV):
        feat_test = build_statement_features(pd.read_csv(TEST_STMT_CSV), df_test, 'id','record_time')
    else:
        feat_test = pd.DataFrame({'id': df_test['id'].unique()})
    feat_test.to_csv(OUT_TEST_FEAT, index=False, encoding='utf-8')
    print('testaa_statement_feature_v2.csv 保存完成：', OUT_TEST_FEAT, feat_test.shape)
else:
    print('未检测到测试集路径，跳过测试流水特征生成。')

train_statement_feature_v2.csv 保存完成： train/train_statement_feature_v2.csv (53480, 22)
testaa_statement_feature_v2.csv 保存完成： testaa/testaa_statement_feature_v2.csv (20054, 22)


**性能小贴士**：若流水量特别大，可先对异常大额进行 Winsorize（如 1%/99% 截尾），或在构造分位数前做分桶降采样；也可以按月份切分批量处理后再外层聚合。